# Customer Churn Prediction
## Notebook 1 — Exploratory Data Analysis

This notebook covers the first two phases of the CRISP-DM methodology:
**Business Understanding** and **Data Understanding**.

**Dataset:** IBM Telco Customer Churn (7,043 customers, 21 features)  
**Goal:** Identify the key drivers of customer churn and prepare the data for modelling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Missing Values ---')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\n--- TotalCharges sample (object column) ---')
print(df['TotalCharges'].head(10))

In [ ]:
# TotalCharges is stored as object — convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 11 rows have TotalCharges = NaN (new customers with tenure=0)
print(f'Rows with NaN TotalCharges: {df["TotalCharges"].isna().sum()}')
print(df[df['TotalCharges'].isna()][['tenure', 'MonthlyCharges', 'TotalCharges']].head())

# Impute with MonthlyCharges (first month billing)
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])
print('\nMissing values after imputation:', df['TotalCharges'].isna().sum())

## 2. Target Variable — Churn Distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(churn_counts.index, churn_counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Churn Count', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_pct.values, labels=churn_pct.index,
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Target Variable: Customer Churn', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/eda_churn_distribution.png', bbox_inches='tight')
plt.show()

print(f'Class imbalance ratio: {churn_pct["No"]:.1f}% No Churn vs {churn_pct["Yes"]:.1f}% Churn')

## 3. Categorical Features vs Churn

In [ ]:
categorical_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'InternetService', 'Contract',
    'PaymentMethod', 'PaperlessBilling'
]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    churn_rate = df.groupby(col)['Churn'].apply(
        lambda x: (x == 'Yes').mean() * 100
    ).reset_index()
    churn_rate.columns = [col, 'ChurnRate']
    churn_rate = churn_rate.sort_values('ChurnRate', ascending=False)

    bars = axes[i].bar(churn_rate[col].astype(str), churn_rate['ChurnRate'],
                       color=sns.color_palette('RdYlGn_r', len(churn_rate)),
                       edgecolor='white')
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].tick_params(axis='x', rotation=20)
    axes[i].axhline(y=churn_pct['Yes'], color='navy', linestyle='--',
                    linewidth=1.5, label=f'Avg ({churn_pct["Yes"]:.1f}%)')
    axes[i].legend(fontsize=8)
    for bar, val in zip(bars, churn_rate['ChurnRate']):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', fontsize=9)

plt.suptitle('Churn Rate by Categorical Feature', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/eda_categorical_churn.png', bbox_inches='tight')
plt.show()

## 4. Numerical Features vs Churn

In [ ]:
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(numerical_cols):
    for label, color in [('No', '#2ecc71'), ('Yes', '#e74c3c')]:
        axes[i].hist(df[df['Churn'] == label][col],
                     bins=40, alpha=0.6, color=color, label=label, edgecolor='white')
    axes[i].set_title(col, fontsize=13, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].legend(title='Churn')

plt.suptitle('Numerical Features Distribution by Churn', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/eda_numerical_churn.png', bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap

In [ ]:
df_encoded = df.copy()
df_encoded['Churn_bin'] = (df_encoded['Churn'] == 'Yes').astype(int)
df_encoded['SeniorCitizen'] = df_encoded['SeniorCitizen'].astype(int)

num_df = df_encoded[['tenure', 'MonthlyCharges', 'TotalCharges',
                      'SeniorCitizen', 'Churn_bin']]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/eda_correlation.png', bbox_inches='tight')
plt.show()

## 6. Key Findings

| Finding | Detail |
|---|---|
| **Class imbalance** | 73.5% No Churn vs 26.5% Churn — requires balancing strategy |
| **Contract type** | Month-to-month customers churn at ~43%, vs ~3% for two-year contracts |
| **Tenure** | Churners have significantly lower tenure (median ~10 months vs ~38 months) |
| **Internet service** | Fiber optic customers churn at ~42%, DSL at ~19% |
| **Monthly charges** | Churners pay ~$10 more per month on average |
| **Data quality** | 11 missing values in TotalCharges (new customers, tenure=0) — imputed with MonthlyCharges |

These findings directly inform the feature engineering and modelling strategy in Notebook 2.